# Weekly WTI RAG4CTS Overlay Experiment

This notebook applies a RAG4CTS-style retrieval overlay to weekly WTI forecasting.

Core design:
- Target: `Com_CrudeOil`
- Frequency: weekly
- Horizon: 2 weeks
- Folds: recent 48 rolling origins
- Retrieval history: 16 weeks by default
- No future exogenous variables are used
- Candidate future paths are used only after historical candidates are selected
- Optional deep model overlay can use `cv_predictions_all_models.csv` from the weekly fold48 run

Compared methods:
- `target_only_model`: causal Ridge on target-only historical features
- `multivariate_model`: causal Ridge on all historical features
- `retrieval_only`: target-history retrieval only
- `model_plus_retrieval`: multivariate model blended with target retrieval
- `feature_retrieval`: covariate-weighted retrieval
- `feature_model_retrieval`: multivariate model blended with covariate retrieval
- `rag4cts_feature_retrieval`: MI-weighted, shock/bucket-aware, recent-tail-boost retrieval
- `rag4cts_feature_model_retrieval`: multivariate model blended with the improved RAG4CTS retrieval
- if deep model predictions exist: `deep_GRU`, `deep_TimeXer`, `deep_iTransformer`, and retrieval overlays


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

BOOTSTRAP_SENTINEL = Path('/content/newoil_colab_bootstrap_rag4cts_overlay_v2')
PACKAGE_SPECS = [
    'numpy>=1.26,<2.2',
    'pandas==2.2.2',
    'matplotlib>=3.8,<3.11',
    'pyyaml==6.0.2',
]
CRITICAL_IMPORTS = ['numpy', 'pandas', 'matplotlib', 'yaml']


def import_is_healthy(module_name):
    try:
        importlib.import_module(module_name)
        return True
    except Exception as exc:
        print(f'Import check failed for {module_name}: {type(exc).__name__}: {exc}')
        return False

bootstrap_changed = False
healthy = BOOTSTRAP_SENTINEL.exists() and all(import_is_healthy(module_name) for module_name in CRITICAL_IMPORTS)
if not healthy:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', *PACKAGE_SPECS], check=True)
    BOOTSTRAP_SENTINEL.write_text('ok')
    bootstrap_changed = True

sanity_code = """
import numpy, pandas, matplotlib, yaml
print('import sanity ok')
"""
sanity = subprocess.run([sys.executable, '-c', sanity_code], capture_output=True, text=True)
print(sanity.stdout)
if sanity.returncode != 0:
    print(sanity.stderr)
    raise RuntimeError('Dependency sanity check failed. Runtime restart may be required.')

if bootstrap_changed:
    print('Dependencies changed. If Colab behaves oddly, Runtime > Restart runtime, then run all cells again.')

try:
    import torch
    print('torch cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu count:', torch.cuda.device_count())
        print('gpu name:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch gpu check skipped:', type(exc).__name__, exc)


In [ ]:
WORKDIR = Path('/content/newoil')
REPO_URL = 'https://github.com/Jaeho777/newoil.git'

if WORKDIR.exists():
    subprocess.run(['git', '-C', str(WORKDIR), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)

commit = subprocess.check_output(['git', '-C', str(WORKDIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Using newoil commit:', commit)


In [ ]:
OUTPUT_ROOT = Path('/content/drive/MyDrive/newoil_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# If True, this first runs the full GRU/TimeXer/iTransformer weekly fold48 model CV.
# It can take hours. If False, the notebook uses the latest existing fold48 model run if available.
RUN_DEEP_MODEL_CV = False
USE_LATEST_DEEP_MODEL_RUN_IF_AVAILABLE = True

HISTORY_LENGTH = 16
HORIZON = 2
N_FOLDS = 48
STEP_SIZE = 1
TOP_K = 12
BLEND_ALPHA = 0.5

# RAG4CTS guide additions: stronger recent signal, tail-aware MI, and bucket-aware retrieval.
TAIL_BOOST_WEEKS = 3
TAIL_BOOST = 2.0
SHOCK_QUANTILE = 0.85
TAIL_MI_QUANTILE = 0.80
EVENT_QUANTILE = 0.10

print('RUN_DEEP_MODEL_CV:', RUN_DEEP_MODEL_CV)
print('GPU note: RUN_DEEP_MODEL_CV=False runs only RAG/Ridge/retrieval baselines, so GPU is not used or needed.')
print('GPU note: set RUN_DEEP_MODEL_CV=True to train GRU/TimeXer/iTransformer on GPU before applying retrieval overlay.')
print('OUTPUT_ROOT:', OUTPUT_ROOT)


In [ ]:
deep_predictions_path = None

if RUN_DEEP_MODEL_CV:
    deep_packages = [
        'rich>=13,<15',
        'protobuf>=4.25,<6',
        'tensorboard>=2.18,<2.20',
        'utilsforecast',
        'coreforecast',
        'lightning-utilities>=0.11,<0.16',
        'torchmetrics>=1.6,<1.9',
        'pytorch-lightning>=2.4,<2.6',
        'ray[tune]>=2.20,<3.0',
        'neuralforecast==3.1.7',
    ]
    optional_torch_packages = [package for package in ['torchvision', 'torchaudio'] if importlib.util.find_spec(package) is not None]
    if optional_torch_packages:
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', *optional_torch_packages], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', *deep_packages], check=True)
    subprocess.run(['nvidia-smi'], check=False)
    subprocess.run([sys.executable, '-c', 'import torch, pytorch_lightning, ray, neuralforecast; print("deep import sanity ok")'], check=True)
    deep_cmd = [
        sys.executable, '-u',
        str(WORKDIR / 'scripts' / 'run_weekly_logret_h2_fold48.py'),
        '--repo-root', str(WORKDIR),
        '--output-root', str(OUTPUT_ROOT),
        '--horizon', str(HORIZON),
        '--n-windows', str(N_FOLDS),
        '--step-size', str(STEP_SIZE),
        '--val-size', '48',
        '--input-size', '64',
        '--max-epochs', '500',
        '--devices', '1',
    ]
    print('RUN DEEP MODEL CV:', ' '.join(deep_cmd), flush=True)
    subprocess.run(deep_cmd, check=True)

if USE_LATEST_DEEP_MODEL_RUN_IF_AVAILABLE:
    model_runs = sorted(OUTPUT_ROOT.glob('weekly_logret_h2_fold48_*'), key=lambda p: p.stat().st_mtime)
    if model_runs:
        candidate = model_runs[-1] / 'cv_predictions_all_models.csv'
        if candidate.exists():
            deep_predictions_path = candidate
            print('Using deep model predictions:', deep_predictions_path)
        else:
            print('Latest model run has no cv_predictions_all_models.csv:', model_runs[-1])
    else:
        print('No previous weekly_logret_h2_fold48_* run found. RAG notebook will run lightweight model/retrieval baselines only.')


In [ ]:
rag_cmd = [
    sys.executable, '-u',
    str(WORKDIR / 'scripts' / 'run_weekly_rag4cts_overlay.py'),
    '--repo-root', str(WORKDIR),
    '--output-root', str(OUTPUT_ROOT),
    '--history-length', str(HISTORY_LENGTH),
    '--horizon', str(HORIZON),
    '--n-folds', str(N_FOLDS),
    '--step-size', str(STEP_SIZE),
    '--top-k', str(TOP_K),
    '--blend-alpha', str(BLEND_ALPHA),
    '--tail-boost-weeks', str(TAIL_BOOST_WEEKS),
    '--tail-boost', str(TAIL_BOOST),
    '--shock-quantile', str(SHOCK_QUANTILE),
    '--tail-mi-quantile', str(TAIL_MI_QUANTILE),
    '--event-quantile', str(EVENT_QUANTILE),
]
if deep_predictions_path is not None:
    rag_cmd.extend(['--model-predictions', str(deep_predictions_path)])

print('RUN RAG4CTS OVERLAY:', ' '.join(rag_cmd), flush=True)
subprocess.run(rag_cmd, check=True)


In [ ]:
import pandas as pd
from IPython.display import Image, display, Markdown

runs = sorted(OUTPUT_ROOT.glob('weekly_rag4cts_overlay_*'), key=lambda p: p.stat().st_mtime)
latest = runs[-1]
print('Latest RAG output:', latest)

config = pd.read_json(latest / 'config.json', typ='series')
display(config)

metrics = pd.read_csv(latest / 'tables' / 'metrics_summary.csv')
print('\nMetrics summary')
display(metrics)

regime_summary = pd.read_csv(latest / 'tables' / 'regime_summary.csv')
print('\nRegime/tail-event summary')
display(regime_summary)

per_fold = pd.read_csv(latest / 'tables' / 'per_fold_metrics.csv')
print('\nPer-fold diagnostics: MCR and extreme underprediction')
display(per_fold.sort_values(['regime', 'price_MAPE']).head(30))

leakage = pd.read_csv(latest / 'tables' / 'leakage_audit.csv')
print('\nLeakage audit')
display(leakage)
if not leakage.drop(columns=['fold', 'origin_dt'], errors='ignore').all().all():
    raise RuntimeError('Leakage audit failed. Inspect tables/leakage_audit.csv before using results.')

retrieval = pd.read_csv(latest / 'tables' / 'retrieval_candidates.csv')
print('\nTop retrieval candidates')
display(retrieval.head(20))

display(Image(filename=str(latest / 'plots' / 'metrics_price_mape.png')))
display(Image(filename=str(latest / 'plots' / 'tail_mcr_extreme_underprediction.png')))
display(Image(filename=str(latest / 'plots' / 'continuous_forecast_overlay.png')))
display(Markdown((latest / 'report.md').read_text()))
